# AI Engineering — Topic 3: V1 Document Data Pipeline

**Project:** SortSmart AI — Document-Grounded Household Waste Sorting Assistant  
**Student:** Lai Yan Qi (A24AI0109) · **Course:** AI Engineering · **Date:** 11 November 2026  
**Repository:** https://github.com/YANKEY-CODE/ai-engineering-sortsmart-ai

**Selected pathway:** A — PDF / TXT. Week 2 explicitly scopes SortSmart to *text-described* waste questions, **not computer vision**. Requirements traceability: **FR2, RAI1, RAI3, NFR3** (source evidence, provenance, freshness and reproducibility).

> **IMPORTANT — SYNTHETIC PRACTICE DATA, NOT DOMAIN VALIDATION.** The course-supplied PDF and TXT concern fictional student policies, not waste sorting. This notebook demonstrates document ingestion, parsing, comparison, validation and lineage only. It does **not** implement an LLM, validate disposal guidance, or measure the intended AI model's performance. Replace these files with approved, current Malaysian guidance before using prepared text in SortSmart AI. Public redistribution of the course pack has **not** been verified.

**Run order:** Run all cells top-to-bottom. In Colab, upload `Topic_3_Dummy_Datasets.zip` using the upload cell below or place it at `/content/week3_project/data/raw/Topic_3_Dummy_Datasets.zip`. No API keys are used. Colab's `/content` storage is temporary.

## Part A — Requirements, workspace, sources and ingestion

In [ ]:
# Cell 1 — Common workspace (always run first)
import os, json, hashlib, re, shutil, subprocess, sys
from pathlib import Path
from datetime import datetime, timezone

BASE_DIR = Path(os.environ.get('WEEK3_PROJECT_ROOT', '/content/week3_project')).resolve()
RAW_DIR = BASE_DIR / 'data' / 'raw'
PROCESSED_DIR = BASE_DIR / 'data' / 'processed'
PIPELINE_DIR = BASE_DIR / 'data' / 'pipeline_v1' / 'documents'
for d in (RAW_DIR, PROCESSED_DIR, PIPELINE_DIR):
    d.mkdir(parents=True, exist_ok=True)
RAW, PROCESSED, BASE, OUT = RAW_DIR, PROCESSED_DIR, BASE_DIR, PROCESSED_DIR

def sha256_file(path):
    digest=hashlib.sha256()
    with Path(path).open('rb') as stream:
        for piece in iter(lambda: stream.read(65536), b''):
            digest.update(piece)
    return digest.hexdigest()

def utc_now():
    return datetime.now(timezone.utc).isoformat()

print('Workspace:', BASE_DIR)
print('Raw:', RAW_DIR)
print('Processed:', PROCESSED_DIR)
print('Pipeline:', PIPELINE_DIR)


In [ ]:
# Cell 2 — Optional Colab ZIP upload + safe extraction of only the two document examples
# For Colab: first upload the provided ZIP with this cell if it is not already in data/raw.
from zipfile import ZipFile

EXPECTED_SOURCE_FILES = [
    'topic3_dummy_datasets/01_documents/student_handbook_demo.pdf',
    'topic3_dummy_datasets/01_documents/faq_demo.txt',
]

candidates = [RAW_DIR / 'Topic_3_Dummy_Datasets.zip',
              Path('/content/Topic_3_Dummy_Datasets.zip')]
if os.environ.get('WEEK3_DUMMY_ZIP'):
    candidates.insert(0, Path(os.environ['WEEK3_DUMMY_ZIP']))
zip_path = next((p for p in candidates if p.is_file()), None)
if zip_path is None and not all((RAW_DIR / s).is_file() for s in EXPECTED_SOURCE_FILES):
    try:
        from google.colab import files
        print('Choose Topic_3_Dummy_Datasets.zip from your computer.')
        uploaded = files.upload()
        if 'Topic_3_Dummy_Datasets.zip' in uploaded:
            zip_path = RAW_DIR / 'Topic_3_Dummy_Datasets.zip'
            zip_path.write_bytes(uploaded['Topic_3_Dummy_Datasets.zip'])
    except ModuleNotFoundError:
        pass

if zip_path is not None:
    with ZipFile(zip_path) as zf:
        for relative in EXPECTED_SOURCE_FILES:
            if relative not in zf.namelist():
                raise FileNotFoundError(f'Missing from ZIP: {relative}')
            destination = RAW_DIR / relative  # explicit allow-list; no unsafe extractall()
            destination.parent.mkdir(parents=True, exist_ok=True)
            destination.write_bytes(zf.read(relative))
    print('Extracted selected document files from:', zip_path)

missing = [s for s in EXPECTED_SOURCE_FILES if not (RAW_DIR/s).is_file()]
assert not missing, 'Upload the supplied Topic 3 ZIP; missing: '+repr(missing)
print('Verified sources:')
for relative in EXPECTED_SOURCE_FILES:
    p=RAW_DIR/relative
    print(' ',relative,'| bytes:',p.stat().st_size,'| SHA-256:',sha256_file(p)[:16]+'…')


In [ ]:
# Cell 3 — Explicit source inventory + requirement mapping
SOURCE_SPECS = [
  {'source_id':'S01', 'relative_path':EXPECTED_SOURCE_FILES[0], 'media_type':'application/pdf',
   'version':'2026 Demo Edition (as printed in PDF)', 'owner':'Teaching Team, synthetic course pack',
   'rights':'Course-provided synthetic practice; public redistribution not verified',
   'jurisdiction':'not applicable', 'domain_relevance':'TECHNIQUE ONLY — not waste guidance'},
  {'source_id':'S02', 'relative_path':EXPECTED_SOURCE_FILES[1], 'media_type':'text/plain',
   'version':'undated synthetic FAQ', 'owner':'Topic 3 teaching pack (author not stated in file)',
   'rights':'Course-provided synthetic practice; public redistribution not verified',
   'jurisdiction':'not applicable', 'domain_relevance':'TECHNIQUE ONLY — not waste guidance'},
]
REQUIREMENTS = ['FR2','RAI1','RAI3','NFR3']
for spec in SOURCE_SPECS:
    source = RAW_DIR / spec['relative_path']
    assert source.is_file() and source.stat().st_size > 0
    print(spec['source_id'], '|', source.name, '|', spec['media_type'],
          '|', source.stat().st_size,'bytes', '|', spec['domain_relevance'])
print('Mapped Week 2 requirements:', ', '.join(REQUIREMENTS))


In [ ]:
# Cell 4 — Document parser + inspection of two distinct files
try:
    from pypdf import PdfReader
except ModuleNotFoundError:
    subprocess.check_call([sys.executable, '-m', 'pip', '-q', 'install', 'pypdf'])
    from pypdf import PdfReader

def load_segments(spec):
    p = RAW_DIR/spec['relative_path']
    if not p.is_file():
        raise FileNotFoundError(f'Input missing: {p}')
    if p.suffix.lower()=='.pdf':
        reader=PdfReader(str(p))
        segments=[{'page':i+1,'text':page.extract_text() or ''}
                  for i,page in enumerate(reader.pages)]
    elif p.suffix.lower()=='.txt':
        segments=[{'page':None,'text':p.read_text(encoding='utf-8-sig')}]
    else:
        raise ValueError('Only PDF and TXT are selected')
    if not segments or not any(s['text'].strip() for s in segments):
        raise ValueError(f'No readable extracted text: {p}')
    return segments

def clean_text(value):
    # Preserve newlines for section/paragraph structure; do not remove source content.
    value=value.replace('\r\n','\n').replace('\r','\n')
    value=re.sub(r'[ \t]+',' ',value)
    value=re.sub(r'\n{3,}', '\n\n',value)
    return value.strip()

source_inspection=[]
for spec in SOURCE_SPECS:
    segments=load_segments(spec)
    cleaned=[clean_text(s['text']) for s in segments]
    item={'source_id':spec['source_id'], 'relative_path':spec['relative_path'],
          'segments':len(segments),'characters':sum(len(t) for t in cleaned),
          'sha256':sha256_file(RAW_DIR/spec['relative_path']),
          'preview':cleaned[0][:160]}
    source_inspection.append(item)
    print('\n',item['source_id'],'—',Path(spec['relative_path']).name)
    print('Segments:', item['segments'],'| readable chars:',item['characters'])
    print('Representative content:',repr(item['preview']))
assert len(source_inspection)==2


## Part B — Compare two chunking methods, prepare, validate and record lineage

In [ ]:
# Cell 5A1 — Chunking candidates (text/document pathway only)
CHUNK_SIZE = 160  # small because the supplied practice texts are short
CHOSEN_METHOD = 'paragraph-aware'

def fixed_chunks(text, limit=CHUNK_SIZE):
    if limit < 1: raise ValueError('chunk size must be positive')
    return [text[i:i+limit] for i in range(0,len(text),limit)] if text else []

def structural_blocks(text):
    # Keep each PDF "Section N" with its following statement, and FAQ Q/A groups.
    # These cues come from inspecting the TWO synthetic files, not from unseen waste documents.
    text=clean_text(text)
    if not text: return []
    parts=re.split(r'\n(?=(?:Section\s+\d+\s*:|Q\s*:))',text,flags=re.I)
    blocks=[]
    for part in parts:
        blocks.extend([p.strip() for p in re.split(r'\n\s*\n',part) if p.strip()])
    return blocks

def pack_chunks(text, limit=CHUNK_SIZE):
    if limit < 1: raise ValueError('chunk size must be positive')
    blocks=structural_blocks(text)
    chunks=[]
    current=''
    for block in blocks:
        # Very long blocks are split in a documented fallback.
        pieces=[block] if len(block)<=limit else fixed_chunks(block,limit)
        for piece in pieces:
            tentative=current+('\n\n' if current else '')+piece
            if current and len(tentative)>limit:
                chunks.append(current)
                current=piece
            else:
                current=tentative
    if current:chunks.append(current)
    return chunks

comparison=[]
for spec in SOURCE_SPECS:
    segments=load_segments(spec)
    for segment in segments:
        clean=clean_text(segment['text'])
        fixed=fixed_chunks(clean)
        structured=pack_chunks(clean)
        assert ''.join(fixed)==clean
        assert fixed and structured
        assert all(0<len(c)<=CHUNK_SIZE for c in structured)
        comparison.append({'source_id':spec['source_id'],'page':segment['page'],
                           'clean_characters':len(clean),'fixed_chunks':len(fixed),
                           'paragraph_chunks':len(structured),
                           'fixed_preview':repr(fixed[0][:85]),
                           'paragraph_preview':repr(structured[0][:85])})
        print(spec['source_id'], 'page:',segment['page'],
              '| fixed:',len(fixed),'| paragraph-aware:',len(structured))
        print(' fixed first:',repr(fixed[0][:90]))
        print(' structured first:',repr(structured[0][:90]))
print('Selected:',CHOSEN_METHOD,'| limit:',CHUNK_SIZE)


In [ ]:
# Cell 5A2 — Transform + export a stable, traceable JSONL corpus
# Part B and Part C call the SAME function with the SAME settings.
def build_prepared_records(specs=SOURCE_SPECS,method=CHOSEN_METHOD,limit=CHUNK_SIZE):
    if method not in ('paragraph-aware','fixed'): raise ValueError('unsupported method')
    results=[]
    for spec in specs:
        source=RAW_DIR/spec['relative_path']
        fingerprint=sha256_file(source)
        for seg in load_segments(spec):
            cleaned=clean_text(seg['text'])
            chunks=pack_chunks(cleaned,limit) if method=='paragraph-aware' else fixed_chunks(cleaned,limit)
            for idx,chunk in enumerate(chunks,1):
                if not chunk.strip():continue
                results.append({
                    'record_id':f"{spec['source_id']}-p{seg['page'] or 0}-{idx:04d}",
                    'text':chunk,
                    'source_id':spec['source_id'],
                    'source':spec['relative_path'],
                    'source_sha256':fingerprint,
                    'page':seg['page'],
                    'source_version':spec['version'],
                    'source_owner':spec['owner'],
                    'source_rights':spec['rights'],
                    'jurisdiction':spec['jurisdiction'],
                    'corpus_status':'SYNTHETIC_PRACTICE_ONLY',
                    'approved_for_end_user_guidance':False,
                    'processing':f'{method}-char-{limit}',
                })
    if not results:raise ValueError('No text records prepared')
    return results

def write_jsonl(path,items):
    path=Path(path)
    path.parent.mkdir(parents=True,exist_ok=True)
    # deterministic output: no run timestamp inside individual data records
    with path.open('w',encoding='utf-8',newline='\n') as out:
        for item in items:
            out.write(json.dumps(item,ensure_ascii=False,sort_keys=True)+'\n')
    return path

prepared_records=build_prepared_records()
prepared_path=write_jsonl(PROCESSED_DIR/'prepared_text.jsonl',prepared_records)
(PROCESSED_DIR/'chunking_comparison.json').write_text(
    json.dumps({'max_chars':CHUNK_SIZE,'selected':CHOSEN_METHOD,'comparisons':comparison},
               indent=2,ensure_ascii=False)+'\n',encoding='utf-8')
print('Part B records:',len(prepared_records),'| prepared:',prepared_path)
print('First record:',prepared_records[0])


In [ ]:
# Cell B3 — Required output contract + content-level checks
def verify_prepared(path,expected_sources=None):
    path=Path(path)
    if not path.is_file() or path.stat().st_size==0:
        raise ValueError(f'Missing or empty prepared output: {path}')
    with path.open(encoding='utf-8') as stream:
        parsed=[json.loads(line) for line in stream if line.strip()]
    if not parsed:raise ValueError('No JSONL records')
    if len({r['record_id'] for r in parsed})!=len(parsed):raise ValueError('Duplicate record IDs')
    for r in parsed:
        if not r['text'].strip() or len(r['text'])>CHUNK_SIZE:
            raise ValueError('Empty/oversized chunk')
        if not r['source_sha256'] or not r['source'] or not r['source_id']:
            raise ValueError('Missing lineage field')
        if r['approved_for_end_user_guidance'] or r['corpus_status']!='SYNTHETIC_PRACTICE_ONLY':
            raise ValueError('Unsafe or mislabelled synthetic record')
    if expected_sources is not None and set(r['source_id'] for r in parsed)!=set(expected_sources):
        raise ValueError('Unexpected source coverage')
    return parsed

validated_records=verify_prepared(prepared_path,['S01','S02'])
assert validated_records==prepared_records
print('B3 VALIDATION: PASS | records:',len(validated_records),
      '| bytes:',prepared_path.stat().st_size,'| SHA256:',sha256_file(prepared_path))


In [ ]:
# Cells 4A / B4.1 / 6A / 6B — Common source -> prepared file lineage manifest
lineage_entries=[]
for spec in SOURCE_SPECS:
    matching=[r for r in validated_records if r['source_id']==spec['source_id']]
    if not matching:raise ValueError('Source not represented in output: '+spec['source_id'])
    src=RAW_DIR/spec['relative_path']
    lineage_entries.append({
        'source_id':spec['source_id'],
        'source':spec['relative_path'],
        'source_sha256':sha256_file(src),
        'source_size_bytes':src.stat().st_size,
        'output':'data/processed/prepared_text.jsonl',
        'output_sha256':sha256_file(prepared_path),
        'output_size_bytes':prepared_path.stat().st_size,
        'output_record_ids':[r['record_id'] for r in matching],
        'output_record_count':len(matching),
        'processing':f'{CHOSEN_METHOD}-char-{CHUNK_SIZE}; PDF page IDs retained',
        'provenance_limitation':'Synthetic education sample; no real Malaysian waste policy, approval or publication provenance',
        'source_rights':spec['rights'],
        'requirements':REQUIREMENTS,
    })
manifest={'pipeline_version':'week3-v1','generated_at_utc':utc_now(),
          'project':'SortSmart AI','dataset_status':'synthetic_practice_only',
          'records':lineage_entries}
manifest_path=BASE_DIR/'data'/'week3_manifest.json'
manifest_path.write_text(json.dumps(manifest,indent=2,ensure_ascii=False)+'\n',encoding='utf-8')
manifest_read=json.loads(manifest_path.read_text(encoding='utf-8'))
assert len(manifest_read['records'])==len(SOURCE_SPECS)
for entry in manifest_read['records']:
    assert sha256_file(RAW_DIR/entry['source'])==entry['source_sha256']
    assert sha256_file(prepared_path)==entry['output_sha256']
    assert entry['output_record_count']>0
print('B4 COMMON LINEAGE VALIDATION: PASS | sources:',len(lineage_entries),
      '| manifest:',manifest_path)


## Part C — V1 executable batch pipeline, manifest and repeatability smoke test

In [ ]:
# Cells 7 + 8A — Reusable source ingestion -> transform -> validate -> persist -> provenance
# Reads only the explicitly approved synthetic example paths from Part A.
def execute_v1_document_pipeline():
    records=build_prepared_records(SOURCE_SPECS,CHOSEN_METHOD,CHUNK_SIZE)
    target=PIPELINE_DIR/'prepared_text.jsonl'
    write_jsonl(target,records)
    loaded=verify_prepared(target,['S01','S02'])
    if loaded!=records:raise ValueError('Saved records differ from transformed records')
    sources=[{'source_id':spec['source_id'],
              'path':spec['relative_path'],
              'sha256':sha256_file(RAW_DIR/spec['relative_path']),
              'bytes':(RAW_DIR/spec['relative_path']).stat().st_size}
             for spec in SOURCE_SPECS]
    content_map={spec['source_id']:[r['record_id'] for r in records if r['source_id']==spec['source_id']]
                 for spec in SOURCE_SPECS}
    run_manifest={
        'pipeline_version':'week3-v1','run_at_utc':utc_now(),
        'pathway':'documents_pdf_txt','project':'SortSmart AI',
        'dataset_status':'synthetic_practice_only',
        'source_count':len(sources),'record_count':len(records),
        'configuration':{'method':CHOSEN_METHOD,'chunk_size_chars':CHUNK_SIZE,
                         'source_paths':[s['relative_path'] for s in SOURCE_SPECS],
                         'source_validation':'nonempty + readable'},
        'stage_contracts':{
            'ingest':'two explicit existing and permitted practice source files',
            'inspect':'readable parsed PDF page(s) and one UTF-8 text record',
            'prepare':'nonempty section/Q-A aware <= max length chunks',
            'validate':'unique IDs, source coverage, synthetic status, JSONL reload',
            'store':'data/pipeline_v1/documents/prepared_text.jsonl',
            'record_lineage':'SHA-256 of each source, output, and IDs linked to source'},
        'inputs':sources,
        'output':{'path':'data/pipeline_v1/documents/prepared_text.jsonl',
                  'sha256':sha256_file(target),'bytes':target.stat().st_size,
                  'records':len(records)},
        'source_to_record_ids':content_map,
        'known_limitations':['synthetic student examples are not waste guidance',
                             'no model, embedding or retrieval evaluation',
                             'not proof of licence/redistribution permission',
                             'simple section-aware splitter may not generalise to complex PDFs'],
    }
    (PIPELINE_DIR/'pipeline_manifest.json').write_text(
        json.dumps(run_manifest,indent=2,ensure_ascii=False)+'\n',encoding='utf-8')
    assert sha256_file(target)==sha256_file(prepared_path), 'Part C must match Part B configuration/output'
    print('PIPELINE SUCCESS | sources:',len(sources),'| records:',len(records))
    print('Part C prepared:',target)
    print('Pipeline manifest:',PIPELINE_DIR/'pipeline_manifest.json')
    return run_manifest

first_run=execute_v1_document_pipeline()


In [ ]:
# Cell 9 — Re-execute pipeline and compare stable source + output fingerprints, sizes, counts, mappings
second_run=execute_v1_document_pipeline()
checks={
 'same_source_paths': [s['path'] for s in first_run['inputs']]==[s['path'] for s in second_run['inputs']],
 'same_source_sha256': [s['sha256'] for s in first_run['inputs']]==[s['sha256'] for s in second_run['inputs']],
 'same_source_sizes': [s['bytes'] for s in first_run['inputs']]==[s['bytes'] for s in second_run['inputs']],
 'same_output_sha256':first_run['output']['sha256']==second_run['output']['sha256'],
 'same_output_bytes':first_run['output']['bytes']==second_run['output']['bytes'],
 'same_record_count':first_run['record_count']==second_run['record_count'],
 'same_source_record_mapping':first_run['source_to_record_ids']==second_run['source_to_record_ids'],
 'matches_part_b_export':first_run['output']['sha256']==sha256_file(prepared_path),
}
assert all(checks.values()),'Repeatability check failed: '+str(checks)
repeatability_report={
    'test':'week3_v1_repeatability_smoke_test','checked_at_utc':utc_now(),
    'result':'PASS','checks':checks,'source_count':second_run['source_count'],
    'record_count':second_run['record_count'],
    'prepared_sha256':second_run['output']['sha256'],
    'note':'Two runs in one environment with the same sources/config; not cross-environment reproducibility or model quality.'
}
(PIPELINE_DIR/'repeatability_smoke_test.json').write_text(
    json.dumps(repeatability_report,indent=2,ensure_ascii=False)+'\n',encoding='utf-8')
print('CELL 9 REPEATABILITY: PASS')
for name,ok in checks.items():print('PASS' if ok else 'FAIL',name)
print('Evidence:',PIPELINE_DIR/'repeatability_smoke_test.json')


In [ ]:
# Final checkpoint — inspect actual files and sample output; no AI-domain claims
print('\n=== TOPIC 3 DELIVERABLES ===')
for p in [prepared_path,PROCESSED_DIR/'chunking_comparison.json',manifest_path,
          PIPELINE_DIR/'prepared_text.jsonl',PIPELINE_DIR/'pipeline_manifest.json',
          PIPELINE_DIR/'repeatability_smoke_test.json']:
    assert p.is_file() and p.stat().st_size > 0
    print(p.relative_to(BASE_DIR),'|',p.stat().st_size,'bytes')
print('\nExample processed chunk:',repr(prepared_records[0]['text']))
print('DONE — pipeline exercise validated; SortSmart AI application is NOT implemented here.')


### How to use the result later

- The pipeline code is for **document ingestion only** and does not implement FR1 classification, answering, citation truthfulness, retrieval, safety triage, or production deployment.
- Before Topic 4, replace both synthetic input files with permitted, manually reviewed **JPSPN/DOE/KPKT** material relevant to real household waste queries; record source URLs, publication/retrieval dates, jurisdiction, rights and approval status.
- The `page` property is meaningful for the PDF example. Real PDF extraction may lose tables, headers, structure, footnotes or image text; inspect before using. DOCX/webpages require different loader/conversion steps.
- **Do not upload restricted raw PDFs or data** to a public GitHub repository. Share only authorised outputs and a non-sensitive execution record; these files are class-created examples with redistribution rights unverified.
- The notebook is repeatable after the source ZIP is supplied, and the tests here show same-input/same-environment determinism only.